# Laborator 1

## Cerințe inițiale laborator

Orice mediu de dezvoltare Python:
- PyCharm: https://www.jetbrains.com/pycharm/
- Spyder: https://www.spyder-ide.org/
- colab: https://colab.research.google.com/
- VSCode: https://code.visualstudio.com/
- etc

NumPy: https://numpy.org/install/

scikit-learn: https://scikit-learn.org/stable/install.html

CUDA (doar pentru PyTorch GPU): https://developer.nvidia.com/cuda-downloads/

PyTorch: https://pytorch.org/get-started/locally/

## Exerciții

### 1. Scalare Min-Max
Fie un vector tensor X de dimensiune necunoscută.

Scalați elementele acestuia în intervalul [0, 1].

In [2]:
import numpy as np
import torch

X = torch.from_numpy(np.random.uniform(size=(3, 4)))
print(X)
print(X.shape)   # Afisati dimensiunea tensorului X
X = (X - X.min()) / (X.max() - X.min())
print(X)

assert torch.allclose(X.amin(), torch.tensor(0.0, dtype=X.dtype))
assert torch.allclose(X.amax(), torch.tensor(1.0, dtype=X.dtype))

tensor([[9.8773e-01, 7.6797e-01, 3.6537e-01, 4.0743e-01],
        [1.9619e-01, 3.4062e-02, 7.8990e-01, 2.7904e-01],
        [3.9704e-04, 5.4158e-01, 7.3881e-01, 9.3148e-01]], dtype=torch.float64)
torch.Size([3, 4])
tensor([[1.0000, 0.7774, 0.3697, 0.4123],
        [0.1983, 0.0341, 0.7996, 0.2822],
        [0.0000, 0.5481, 0.7479, 0.9430]], dtype=torch.float64)


**Hint**:

`torch.min(...)` / `tensor.min(...)`

`torch.max(...)` / `tensor.max(...)`

### 2. Standardizare — medie 0, varianță 1 (vector)
Fie un vector tensor X de dimensiune necunoscută.

Normalizați elementele acestuia astfel încât să aibe medie 0 și varianță 1.

In [3]:
X = torch.from_numpy(np.random.uniform(size=(3, 4)))
print(X)
X = ( X - X.mean() ) / X.std()
print(X)

assert torch.allclose(X.mean(), torch.tensor(0.0, dtype=X.dtype), atol=1e-6)
assert torch.allclose(X.var(), torch.tensor(1.0, dtype=X.dtype), atol=1e-6)

tensor([[0.2741, 0.0111, 0.9058, 0.8070],
        [0.0367, 0.1631, 0.6962, 0.3479],
        [0.4851, 0.4751, 0.3885, 0.8652]], dtype=torch.float64)
tensor([[-0.5816, -1.4284,  1.4528,  1.1347],
        [-1.3458, -0.9390,  0.7779, -0.3437],
        [ 0.0981,  0.0659, -0.2131,  1.3220]], dtype=torch.float64)


**Hint**:

`torch.mean(...)` / `tensor.mean(...)`

`torch.std(...)` / `tensor.std(...)`

`torch.var(...)` / `tensor.var(...)`

### 3. Scalare Min-Max pe coloane (matrice)
Fie o matrice tensor X de dimensiune necunoscută.

Scalați elementele fiecărei coloane în intervalul [0, 1].

In [4]:
X = torch.from_numpy(np.random.uniform(size=(3, 4)))
print(X)
X = ( X - X.min(dim=0).values ) / ( X.max(dim=0).values - X.min(dim=0).values )
print(X)

assert torch.allclose(X.amin(dim=0), torch.zeros(X.shape[1], dtype=X.dtype))
assert torch.allclose(X.amax(dim=0), torch.ones(X.shape[1], dtype=X.dtype))

tensor([[0.9194, 0.2877, 0.7810, 0.2530],
        [0.3784, 0.1723, 0.1408, 0.5876],
        [0.7061, 0.3217, 0.5499, 0.7809]], dtype=torch.float64)
tensor([[1.0000, 0.7724, 1.0000, 0.0000],
        [0.0000, 0.0000, 0.0000, 0.6338],
        [0.6058, 1.0000, 0.6390, 1.0000]], dtype=torch.float64)


**Hint**:

Similar cu np, se pot realiza reduceri parțiale (pe linie / coloană).

Similar cu np, se pot realiza operații între matrice și vector.

### 4. Standardizare pe coloane (matrice)
Fie o matrice tensor X de dimensiune necunoscută.

Normalizați elementele fiecărei coloane astfel încât să aibe medie 0 și varianță 1.

In [5]:
X = torch.from_numpy(np.random.uniform(size=(3, 4)))
print(X)
X = ( X - X.mean(dim=0) ) / X.std(dim=0)
print(X)

assert torch.allclose(X.mean(dim=0), torch.zeros(X.shape[1], dtype=X.dtype), atol=1e-6)
assert torch.allclose(X.var(dim=0), torch.ones(X.shape[1], dtype=X.dtype), atol=1e-6)

tensor([[0.6866, 0.2444, 0.6295, 0.4610],
        [0.6734, 0.2726, 0.3606, 0.3749],
        [0.0297, 0.6716, 0.3673, 0.1178]], dtype=torch.float64)
tensor([[ 0.5948, -0.6353,  1.1544,  0.8015],
        [ 0.5597, -0.5174, -0.5992,  0.3191],
        [-1.1545,  1.1527, -0.5552, -1.1206]], dtype=torch.float64)


### 5. Matricea de covarianță a trăsăturilor
Fie o matrice tensor X de dimensiune necunoscută.

Fiecare linie a matricii reprezintă un exemplu și fiecare coloană o trăsătură.

* Calculați matricea de covarință a trăsăturilor K.

$$K = \frac{1}{n-1} X_c^\top X_c, \qquad X_c = X - \bar{X}$$

unde $X_c$ este matricea centrată (fiecare coloană are media 0) și $n$ este numărul de exemple (linii ale lui $X$).

* Element cu element:

$$K_{ij} = \frac{1}{n-1} \sum_{k=1}^{n} (x_{ki} - \bar{x}_i)(x_{kj} - \bar{x}_j)$$

In [7]:
X = torch.from_numpy(np.random.uniform(size = (3, 4)))

Xc = X - X.mean(dim=0)

K = 1 / ( X.shape[0] - 1 ) * (Xc.T @ Xc)
print(K)

assert torch.allclose(K, K.T, atol=1e-6)  # simetrică
assert torch.allclose(K, torch.cov(X.T), atol=1e-4)  # comparație cu funcția din bibliotecă

tensor([[ 0.0322, -0.0135, -0.0277, -0.0314],
        [-0.0135,  0.0061,  0.0080,  0.0093],
        [-0.0277,  0.0080,  0.0540,  0.0593],
        [-0.0314,  0.0093,  0.0593,  0.0650]], dtype=torch.float64)


**Hint**:

* K[i, j] = covarianța între trăsătura i și j.

* Încercați să calculați K direct ca o operație matriceală.

### 6. Descompunere în valori/vectori proprii și proiecție PCA
Fie o matrice tensor X de dimensiune necunoscută.

Fiecare linie a matricii reprezintă un exemplu și fiecare coloană o trăsătură.

1. Calculați vectorii proprii V și valorile proprii D pentru matricea de covarianță a trăsăturilor K.

2. Sortați descrescător valorile proprii. Rearanjați vectorii proprii după valorile sortate.

3. Realizați proiecția PCA pe un număr de trăsături ales.

In [ ]:
X = torch.from_numpy(np.random.uniform(size = (3, 4)))

Xc = X - X.mean(dim=0)

K = 1 / ( X.shape[0] - 1 ) * (Xc.T @ Xc)

D, V = torch.linalg.eigh(K)

# sortare descrescătoare a valorilor proprii și rearanjarea vectorilor proprii
D = torch.flip(D, dims=[0])
V = torch.flip(V, dims=[1])
# verificare: V @ diag(D) @ inv(V) == K


# proiecție PCA pe primele i trăsături
i = 2
X_PCA = Xc @ V[:,:i]

assert torch.allclose(K, V @ torch.diag(D) @ torch.linalg.inv(V), atol=1e-4)
assert torch.all(D[:-1] >= D[1:])  # valorile proprii sunt sortate descrescător
assert torch.allclose(V.T @ V, torch.eye(V.shape[0], dtype=V.dtype), atol=1e-4)  # vectori proprii ortonormali
assert X_PCA.shape == (X.shape[0], i)

**Hint**:

`torch.linalg.eigh(...)` / `torch.linalg.eig(...)`

`torch.argsort(...)`

Rearanjarea a fost făcută cu succes dacă V D V<sup>-1</sup> = K, unde D este matricea diagonală cu valorile proprii pe diagonală.

`torch.diag(...)`

`torch.linalg.inv(...)`

`torch.allclose(...)`

X_PCA = X_centrat · V<sub>i</sub>, unde V<sub>i</sub> reprezintă matricea vectorilor proprii cu toate liniile si primele i coloane (cele mai importante i trăsături).

### 7. Proiecție PCA albită (whitening)
Fie o matrice tensor X de dimensiune necunoscută.

Fiecare linie a matricii reprezintă un exemplu și fiecare coloană o trăsătură.

Calculați proiecția PCA albită pe un număr de trăsături.

In [15]:
print(V.shape, D.shape)

torch.Size([4, 4]) torch.Size([4])


In [16]:
i = 2
X_PCA_white = Xc @ V[:,:i] @ torch.diag( D[:i] ** (-0.5) )

# covarianța trăsăturilor albite trebuie să fie ~identitate
K_white = (X_PCA_white.T @ X_PCA_white) / (X_PCA_white.shape[0] - 1)
assert torch.allclose(K_white, torch.eye(i, dtype=K_white.dtype), atol=1e-4)

**Hint**:

X_PCA_white = X_centrat · V<sub>i</sub> · D<sub>i</sub><sup>-0.5</sup>, unde D<sub>i</sub> este matricea pătratică cu primele i valori proprii pe diagonală.

### 8. Clasificare Bayes pe setul de date Iris (PCA și PCA albit)
Considerați setul de date iris, cu 4 trăsături și 3 clase, 50 de exemple per clasă.

1. Normalizați fiecare trăsătură la medie 0 și varianță 1.

2. Calculați matricea de covarianță K pentru trăsături.

3. Calculați vectorii proprii V și valorile proprii D pentru K.

4. Pentru cele mai importante 4, 3, 2, 1 trăsături realizați proiecția PCA și PCA albită.

5. Împărțiti setul de date în antrenare (40 de exemple pentru fiecare clasă) și testare (restul de 10 exemple pentru fiecare clasă). Puteți realiza împărțirea și la început cât timp folosiți toate datele pentru preprocesare.

6. Realizați o clasificare Bayes. Presupuneți toate trăsăturile ca având distribuție normală. Calculați acuratețea clasificării.

In [ ]:
import torch
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split

iris = load_iris()
X = torch.tensor(iris.data, dtype=torch.float32)
y = torch.tensor(iris.target)

**Hint**:

`train_test_split(...)` — apelați-l separat pentru fiecare clasă, apoi concatenați rezultatele cu `torch.cat`.

Pentru stabilitate numerică, calculați probabilitățile sub logaritm.

`torch.sum(...)` / `torch.log(...)` / `torch.argmax(...)`

`torch.linalg.inv(...)` / `torch.logdet(...)`

Media pe clasă: `X_train[y_train == c].mean(dim=0)`.

Prior-ul unei clase: proporția exemplelor din `X_train` care aparțin acelei clase — `(y_train == c).float().mean()`.

In [ ]:
# Standardizați X_train și X_test la medie 0 / varianță 1 (ca la ex. 4).
# Important: media și deviația standard se calculează DOAR pe X_train,
# apoi se aplică (aceleași valori) și pe X_test — altfel "vedeți" datele
# de test încă din faza de antrenare (data leakage).
X_mean, X_std = ..., ...
X_train = ...
X_test = ...

assert torch.allclose(X_train.mean(dim=0), torch.zeros(4), atol=1e-6)
assert torch.allclose(X_train.var(dim=0), torch.ones(4), atol=1e-6)

In [ ]:
# Calculați matricea de covarianță K pe X_train (ca la ex. 5) și descompunerea
# proprie (D, V), cu valorile proprii sortate descrescător (ca la ex. 6).
K = ...
D, V = ...

print("Valori proprii:", D)
assert torch.all(D[:-1] >= D[1:])

In [ ]:
def bayes_gaussian_classify(X_train, y_train, X_test):
    """
    Clasificator Bayes: presupune că fiecare clasă are o distribuție normală
    multivariată, cu o matrice de covarianță COMUNĂ pentru toate clasele.

    Pași de urmat:
    1. Pentru fiecare clasă c, calculați media trăsăturilor (mu_c) folosind
       DOAR exemplele din X_train care aparțin clasei c.
    2. Calculați matricea de covarianță comună K_hat (pooled), pe toate
       exemplele din X_train (indiferent de clasă) — la fel ca la ex. 5.
       Adăugați un termen mic pe diagonală (ex: 1e-6 * I) înainte de a o
       inversa, pentru stabilitate numerică.
    3. Calculați probabilitatea a priori a fiecărei clase: proporția
       exemplelor din X_train care aparțin acelei clase.
    4. Pentru fiecare exemplu x din X_test și fiecare clasă c, calculați
       log-verosimilitatea SUB LOGARITM (pentru stabilitate numerică),
       folosind formula densității normale multivariate:

           log N(x | mu_c, K) = -1/2 * (x - mu_c)^T @ K^-1 @ (x - mu_c)
                                 - 1/2 * (d * log(2*pi) + log|K|)

       unde d = numărul de trăsături (X_train.shape[1]).
    5. Adunați log(prior_c) la log-verosimilitate -> log-posterior (nenormalizat).
    6. Alegeți, pentru fiecare exemplu, clasa cu log-posteriorul maxim (argmax).

    Returnează: un tensor cu clasa prezisă pentru fiecare exemplu din X_test.
    """
    classes = torch.unique(y_train)
    n_features = X_train.shape[1]

    # TODO 1: media fiecărei clase (ex: dict {clasă: vector de medii})
    mus = ...

    # TODO 2: covarianța comună (pooled) + inversă + log-determinant
    K_hat = ...
    K_inv = ...
    K_logdet = ...

    # TODO 3: probabilitățile a priori ale claselor
    priors = ...

    # TODO 4-6: pentru fiecare clasă, calculați log-verosimilitatea + log-prior,
    # apoi alegeți argmax peste clase pentru fiecare exemplu din X_test
    log_posteriors = torch.zeros(X_test.shape[0], len(classes))
    for i, c in enumerate(classes):
        diff = ...
        log_likelihood = ...
        log_posteriors[:, i] = ...

    return classes[log_posteriors.argmax(dim=1)]

In [ ]:
# Pentru fiecare număr de trăsături păstrate (1, 2, 3, 4), realizați proiecția
# PCA și proiecția PCA albită (ca la ex. 6 și 7) pe X_train / X_test, apoi rulați
# clasificatorul Bayes definit mai sus și calculați acuratețea pe X_test.
for n_feat in range(1, X.shape[1] + 1):
    Vi, Di = V[:, :n_feat], D[:n_feat]

    X_train_pca = ...
    X_test_pca = ...
    X_train_white = ...
    X_test_white = ...

    for nume, Xtr, Xte in [("PCA", X_train_pca, X_test_pca),
                           ("PCA albit", X_train_white, X_test_white)]:
        y_pred = bayes_gaussian_classify(Xtr, y_train, Xte)
        assert y_pred.shape == y_test.shape
        acc = (y_pred == y_test).float().mean().item()
        print(f"n_features={n_feat} [{nume}]: acuratețe = {acc:.4f}")

**Hint**:

`train_test_split(...)`

Pentru stabilitate numerică, calculați probabilitățile sub logaritm.

`torch.sum(...)` / `torch.prod(...)`

`torch.log(...)`

`torch.argmax(...)`